In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
control_path = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/_control/control_ingestas" 
log_path = "abfss://bronze@stfinbankdevfbcq2026.dfs.core.windows.net/_control/log_ingestas" 
print("rutas configuradas correctamente")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

datos_control = [
    Row(
        nombre_tabla="TB_CLIENTES_CORE",
        tipo_carga_default="Full",
        columna_control=None,
        checkpoint_date=None
    ),
    Row(
        nombre_tabla="TB_PRODUCTOS_CAT",
        tipo_carga_default="Full",
        columna_control=None,
        checkpoint_date=None
    ),
    Row(
        nombre_tabla="TB_SUCURSALES_RED",
        tipo_carga_default="Full",
        columna_control=None,
        checkpoint_date=None
    ),
    Row(
        nombre_tabla="TB_OBLIGACIONES",
        tipo_carga_default="Full",
        columna_control=None,
        checkpoint_date=None
    ),
    Row(
        nombre_tabla="TB_MOV_FINANCIEROS",
        tipo_carga_default="Delta",
        columna_control="fec_mov",
        checkpoint_date=None
    ),
    Row(
        nombre_tabla="TB_COMISIONES_LOG",
        tipo_carga_default="Delta",
        columna_control="fec_cobro",
        checkpoint_date=None
    )
]

schema = StructType([
    StructField("nombre_tabla", StringType(), False),
    StructField("tipo_carga_default", StringType(), False),
    StructField("columna_control", StringType(), True),
    StructField("checkpoint_date", TimestampType(), True)
])

df_control = spark.createDataFrame(datos_control, schema=schema)

df_control.write \
    .format("delta") \
    .mode("overwrite") \
    .save(control_path)

print("tabla control_ingestas creada")

display(df_control)

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,TimestampType,IntegerType

schema_log = StructType([
    StructField("nombre_tabla", StringType(), False),
    StructField("fecha_ejecucion", TimestampType(), False),
    StructField("filas_copiadas", IntegerType(), True),
    StructField("duracion_segundos", IntegerType(), True),
    StructField("estado", StringType(), False),
    StructField("run_id", StringType(), False)
])

df_log_vacio = spark.createDataFrame([], schema_log)

df_log_vacio.write \
    .format("delta") \
    .mode("overwrite") \
    .save(log_path)

print("tabla log_ingestas creada")